## Code to try to compare spectra in QUBIC patch between input maps and reconstructed ones

In [ ]:
# Allows the modification of the files imported without having to restart kernel or re-import them
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import healpy as hp
import pymaster as nmt
import matplotlib.pyplot as plt
from qubic.lib.Qhdf5 import HDF5Dict

In [ ]:
# ─────────────────────────────────────────────
# 1. Chargement de la carte
# ─────────────────────────────────────────────
nside = 128            # Résolution de ta carte
lmax  = 3 * nside - 1  # ℓ_max recommandé (il recommandait 2 * nside)
print("lmax = ", lmax)

type = "parametric"
model = "d0"
instrument = "DB"
foldername = "test_convo_in"
hdf5 = HDF5Dict()
# data = hdf5.load_dict("parametric_d0_DB_test_convo_out_noiseless/Dict/test_None.h5")
data = hdf5.load_dict("{}_{}_{}_{}/Dict/test_None.h5".format(type, model, instrument, foldername))

comp_maps_input = data["maps_in_convolved"]
comp_maps_rec = data["maps"]
comp_maps_notconvolved = data["maps_in"]

In [ ]:
# Charge ta carte (remplace par ton fichier)
i_rec = 0
i_stk = 0
reco_map = comp_maps_rec[i_rec, :, i_stk]
input_map = comp_maps_input[i_rec, :, i_stk]
diff_map = input_map - reco_map

# Ou génère une carte de test
# sky_map = hp.synfast(hp.alm2cl(hp.map2alm(hp.read_map(...))), nside)


In [ ]:
# ─────────────────────────────────────────────
# 2. Création du masque
# ─────────────────────────────────────────────
# Option A : masque à partir d'une calotte galactique
#   → garde les pixels à |b| > b_cut degrés
# b_cut_deg = 20.0
# npix = hp.nside2npix(nside)
# theta, phi = hp.pix2ang(nside, np.arange(npix))   # colatitude / longitude
# b_gal = np.pi / 2 - theta                          # latitude galactique
# mask = (np.abs(b_gal) > np.radians(b_cut_deg)).astype(float)

# Option B : masque à partir d'un disque centré sur (lon, lat)
# lon0, lat0 = 30.0, 45.0          # degrés
# radius_deg = 20.0
# vec0  = hp.ang2vec(np.radians(90 - lat0), np.radians(lon0))
# ipix  = hp.query_disc(nside, vec0, np.radians(radius_deg))
# mask  = np.zeros(npix)
# mask[ipix] = 1.0

# Option C : charge un masque depuis un fichier FITS
# mask = hp.read_map("mon_masque.fits")

# Option D : j'utilise les outils de QUBIC
mask = data["seenpix"]

In [ ]:
# ─────────────────────────────────────────────
# 3. Apodisation du masque (essentielle pour NaMaster)
# ─────────────────────────────────────────────
apod_deg  = 2.0          # taille de l'apodisation en degrés
apod_type = "C2"         # "C1", "C2" ou "Smooth"

mask_apod = nmt.mask_apodization(mask, apod_deg, apotype=apod_type)

# Visualisation optionnelle du masque
hp.mollview(mask_apod, title="Masque apodisé", cmap="RdBu_r")
plt.savefig("masque_apodise.png", dpi=150)
plt.close()

In [ ]:
# ─────────────────────────────────────────────
# 4. Définition des bins en ℓ (bandpowers)
# ─────────────────────────────────────────────
# Bins logarithmiques entre ℓ=2 et lmax
# ell_min, ell_max = 2, lmax
# n_bins = 30

# ells    = np.arange(ell_min, ell_max + 1)
# log_edges = np.geomspace(ell_min, ell_max, n_bins + 1).astype(int)
# log_edges = np.unique(log_edges)          # supprime les doublons éventuels

# bpws  = -1 * np.ones(ell_max + 1, dtype=int)   # -1 = ℓ ignoré
# for i, (l0, l1) in enumerate(zip(log_edges[:-1], log_edges[1:])):
#     bpws[l0:l1] = i

# print("bpws", bpws.astype(np.int32))
# print("ells", ells.astype(np.int32))
# weights = np.ones(len(ells))
# print("weights", weights)
# f_ell = np.ones(len(ells))
# print("f_ell", f_ell)
# print("lmax", lmax)
    
# b = nmt.NmtBin(nside, bpws=bpws, lmax=lmax)
# b = nmt.NmtBin(bpws=bpws, ells=ells, lmax=lmax)

nlb = 20 # 4 modes per bin
b = nmt.NmtBin.from_nside_linear(nside, nlb, is_Dell=False, f_ell=None)

# Alternative simple : bins uniformes de largeur Δℓ
# b = nmt.NmtBin.from_lmax_linear(lmax, nlb=4)

In [ ]:
# ─────────────────────────────────────────────
# 5. Création des champs NaMaster (spin-0)
# ─────────────────────────────────────────────
# Pour un champ scalaire (température, intensité…)
reco_f0 = nmt.NmtField(mask_apod, [reco_map])
input_f0 = nmt.NmtField(mask_apod, [input_map])
diff_f0 = nmt.NmtField(mask_apod, [diff_map])

# Pour deux champs différents (cross-spectrum) :
# f0a = nmt.NmtField(mask_apod, [map_a])
# f0b = nmt.NmtField(mask_apod, [map_b])

In [ ]:
# ─────────────────────────────────────────────
# 6. Calcul de l'espace de travail (workspace)
# ─────────────────────────────────────────────
# Le workspace stocke les matrices de couplage → à sauvegarder/recharger
# pour éviter de les recalculer.
w = nmt.NmtWorkspace()
w.compute_coupling_matrix(input_f0, input_f0, b)

# Sauvegarde / rechargement du workspace (optionnel)
# w.write_to("workspace.fits")
# w.read_from("workspace.fits")

In [ ]:
# ─────────────────────────────────────────────
# 7. Calcul du pseudo-C_ℓ et découplage
# ─────────────────────────────────────────────
reco_cl_coupled   = nmt.compute_coupled_cell(reco_f0, reco_f0)   # pseudo-C_ℓ couplés
reco_cl_decoupled = w.decouple_cell(reco_cl_coupled)         # C_ℓ découplés (bandpowers)

input_cl_coupled   = nmt.compute_coupled_cell(input_f0, input_f0)   # pseudo-C_ℓ couplés
input_cl_decoupled = w.decouple_cell(input_cl_coupled)         # C_ℓ découplés (bandpowers)

diff_cl_coupled   = nmt.compute_coupled_cell(diff_f0, diff_f0)   # pseudo-C_ℓ couplés
diff_cl_decoupled = w.decouple_cell(diff_cl_coupled)         # C_ℓ découplés (bandpowers)

ell_eff = b.get_effective_ells()   # valeurs de ℓ au centre de chaque bin

In [ ]:
cl_full_input_map = hp.anafast(input_map)
full_input_cl = b.bin_cell(cl_full_input_map)

cl_full_reco_map = hp.anafast(reco_map)
full_reco_cl = b.bin_cell(cl_full_reco_map)

In [ ]:
# ─────────────────────────────────────────────
# 8. Correction du bruit (optionnel)
# ─────────────────────────────────────────────
# Si tu as une carte de bruit ou un niveau de bruit uniforme N_ℓ :
# sigma_noise = 1e-6                                # exemple
# nl = sigma_noise**2 * np.ones(lmax + 1)
# nl_coupled   = nmt.compute_coupled_cell(nmt.NmtField(mask_apod, [noise_map]),
#                                         nmt.NmtField(mask_apod, [noise_map]))
# cl_signal    = w.decouple_cell(cl_coupled - nl_coupled)

In [ ]:
# ─────────────────────────────────────────────
# 9. Visualisation
# ─────────────────────────────────────────────
reco_cl = reco_cl_decoupled[0]          # spectre TT (composante [0,0])
input_cl = input_cl_decoupled[0]          # spectre TT (composante [0,0])
diff_cl = diff_cl_decoupled[0]          # spectre TT (composante [0,0])

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(ell_eff, ell_eff * (ell_eff + 1) / (2 * np.pi) * reco_cl,
          "o-", markersize=4, lw=1.5, label="Spectre reconstruit mesuré")
ax.plot(ell_eff, ell_eff * (ell_eff + 1) / (2 * np.pi) * input_cl,
          "o-", markersize=4, lw=1.5, label="Spectre d'entrée mesuré")
# ax.plot(ell_eff, ell_eff * (ell_eff + 1) / (2 * np.pi) * diff_cl,
#           "o-", markersize=4, lw=1.5, label="Spectre des résidus mesuré")
ax.plot(ell_eff, ell_eff * (ell_eff + 1) / (2 * np.pi) * full_input_cl,
          "o-", markersize=4, lw=1.5, label="Spectre en entrée")
ax.plot(ell_eff, ell_eff * (ell_eff + 1) / (2 * np.pi) * full_reco_cl,
          "o-", markersize=4, lw=1.5, label="Spectre carte reco totale")
ax.set_xlabel(r"Multipole $\ell$")
ax.set_ylabel(r"$\ell(\ell+1)\,C_\ell / 2\pi$")
ax.set_title("Spectre de puissance (zone masquée, NaMaster)")
ax.legend()
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.savefig("spectre_namaster.png", dpi=150)
plt.show()

# print("ℓ effectifs :", ell_eff)
# print("C_ℓ         :", cl)